In [ ]:
import torch
import numpy as np
import pandas as pd
import plotly.express as px
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.spatial.distance import mahalanobis
from scipy.linalg import inv, LinAlgError
from sklearn.decomposition import PCA
from sklearn.metrics import davies_bouldin_score
from joblib import Parallel, delayed
import os
import time
TOPICS = {
    "Baseline": None,

    # =========================
    # IMMIGRATION
    # =========================
    "imm_unauth": "unauthorized and undocumented immigrants",
    "birthright": "birthright citizenship",
    "illeg_child": "children of undocumented immigrants",
    "border_wall": "a wall along the U.S.–Mexico border",
    "spend_border": "whether to increase or decrease funding for border security",
    "immig_levels": "the level of immigration into the United States",

    # =========================
    # LABOR / FAMILY POLICY
    # =========================
    "paid_leave": "paid parental or family leave",
    "job_gov_guar": "whether the government should guarantee a job to everyone",
    "min_wage": "whether to change the minimum wage",
    "ft_union": "labor unions",

    # =========================
    # INSTITUTIONS / ACCOUNTABILITY / MEDIA
    # =========================
    "trump_corr": "whether Donald Trump was involved in political corruption",
    "journ_access": "whether journalists should have broad access to government officials and information",
    "checks_power": "to what degree the branches of government should limit each other’s power",
    "rus_interf": "how serious a problem Russian interference in U.S. elections is",
    "ft_sci": "scientists",

    # =========================
    # ELECTIONS / DEMOCRACY
    # =========================
    "voter_id": "whether voters should be required to show identification to vote",
    "felon_vote": "whether people with felony convictions should have the right to vote",
    "vote_denied": "how often people are denied the right to vote",

    # =========================
    # ECONOMY (RETROSPECTIVE)
    # =========================
    "econ_now": "how good or bad the current national economy is",

    # =========================
    # GUNS
    # =========================
    "gun_bkg_chk": "whether background checks should be required for gun purchases",
    "ar_ban": "whether assault weapons should be banned",
    "gun_imp": "to what degree gun regulation is an important political issue",

    # =========================
    # CRIME / POLICING / ORDER
    # =========================
    "death_pen": "whether the death penalty should be used for serious crimes",
    "police_force": "to what degree police should be allowed to use force",
    "urban_unrest": "how serious a problem urban unrest and protests are",
    "ft_police": "the police",
    "biden_crime": "Joe Biden’s handling of crime",
    "crime_spend": "to what degree the government should spend money on dealing with crime",

    # =========================
    # ABORTION / COURTS
    # =========================
    "abortion": "whether abortion should be legal",
    "scotus_abort": "the Supreme Court decisions related to abortion",

    # =========================
    # HEALTH
    # =========================
    "govt_health": "whether the government should provide health insurance",
    "obamacare": "whether the Affordable Care Act should be kept, expanded, or repealed",
    "vax_school": "whether children should be required to be vaccinated to attend school",
    "health_spend": "to what degree the government should spend money to help people pay for health insurance",

    # =========================
    # EDUCATION / SPENDING
    # =========================
    "spend_school": "to what degree the government should spend money on public schools",
    "dei_college": "diversity, equity, and inclusion (DEI) policies on college campuses",
    "affirm_action": "affirmative action in university admissions",

    # =========================
    # REDISTRIBUTION / WELFARE / TAX
    # =========================
    "spend_welfare": "to what degree the government should spend money on welfare programs",
    "spend_poor": "to what degree the government should spend money to help the poor",
    "svc_spend": "to what degree the government should spend money on public services",
    "millionaire_tax": "whether to tax on millionaires",

    # =========================
    # DIVERSITY (formerly Race)
    # =========================
    "assist_black": "to what degree the government should help Black Americans",
    "black_favor": "whether Black people should get special favor",
    "diversity": "to what degree diversity benefits the country",
    "ft_asian": "Asian-Americans",
    "discuss_race": "how often racial issues should be discussed with children",

    # =========================
    # TRANSGENDER
    # =========================
    "trans_bath": "whether transgender people should use bathrooms matching their gender identity",
    "trans_military": "whether transgender people should serve in the U.S. military",
    "ft_trans": "transgender people",

    # =========================
    # LESBIAN / GAY
    # =========================
    "lg_job": "protection for gay and lesbian people from job discrimination",
    "lg_marry": "legal marriage for same-sex couples",
    "lg_refuse_service": "whether businesses should be allowed to refuse service to same-sex couples",

    # =========================
    # GENDER
    # =========================
    "ft_fem": "feminists",

    # =========================
    # CLIMATE / ENVIRONMENT
    # =========================
    "clim_imp": "how important climate change is as an issue",
    "env_bus": "the tradeoff between environmental protection and business interests",
    "ghg_emiss": "whether to regulate greenhouse gas emissions",

    # =========================
    # DEFENSE / FOREIGN POLICY
    # =========================
    "def_spend": "to what degree the government should spend money on national defense",
    "mil_force": "whether the United States should use military force in foreign countries",
    "biden_foreign": "Joe Biden’s handling of foreign relations",
    "israel_aid": "U.S. military assistance to Israel",

    # =========================
    # TRADE
    # =========================
    "free_trade": "free trade agreements with other countries",
    "intl_trade_job": "whether international trade helps or hurts jobs in the United States",
    "limit_imports": "placing new limits on imports",
}







SYSTEM_MSG = (
    "You are simulating the public stance of U.S. politicians.\n\n"
)

# ==========================================
# 1. CONFIGURATION
# ==========================================
# A100 40GB can handle large batches for 8B models
BATCH_SIZE = 128
MODEL_PATH = "/project/jevans/maxzhuyt/models/Meta-Llama-3.1-8B-Instruct"
NOMINATE_CSV = "/project/jevans/maxzhuyt/gss_polarization/data/politicians.csv"

# Topics: Core + Bipartisan/Horseshoe Candidates

# ==========================================
# 2. MODEL LOADER & EXTRACTION (GPU)
# ==========================================
def load_model(path):
    print(f"Loading model from: {path}...")
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    tokenizer = AutoTokenizer.from_pretrained(path, use_fast=True, local_files_only=True)
    
    # CRITICAL: Left padding allows batching without destroying the last token position
    tokenizer.padding_side = 'left' 
    tokenizer.truncation_side = "left"

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    
    model = AutoModelForCausalLM.from_pretrained(
        path, dtype=dtype, device_map="auto", local_files_only=True, attn_implementation="eager"
    )
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    return model, tokenizer

@torch.no_grad()
def extract_heads_batched(model, tokenizer, texts, batch_size=32):
    """
    Optimized extraction for A100.
    """
    model.eval()
    L = model.config.num_hidden_layers
    H = model.config.num_attention_heads
    D_head = model.config.hidden_size // H
    
    activations = []
    
    # Pre-allocate hook containers
    layer_outputs = [None] * L
    
    def get_hook(layer_idx):
        def hook(module, input, output):
            # Input[0] shape: [Batch, Seq, Hidden]
            # Reshape to [Batch, Seq, Heads, Head_Dim]
            # We immediately move to CPU to free VRAM for the next batch
            reshaped = input[0].detach().view(input[0].shape[0], input[0].shape[1], H, D_head)
            layer_outputs[layer_idx] = reshaped[:, -1, :, :].float().cpu().numpy()
        return hook

    # Register hooks once
    hooks = []
    for li in range(L):
        hooks.append(model.model.layers[li].self_attn.o_proj.register_forward_hook(get_hook(li)))

    print(f"  > Extracting with Batch Size {batch_size}...")
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        
        # Fast Tokenization
        formatted_batch = [
            tokenizer.apply_chat_template([{"role": "system", "content": SYSTEM_MSG}, {"role": "user", "content": t}], tokenize=False, add_generation_prompt=True)
            for t in batch
        ]
        
        enc = tokenizer(
            formatted_batch, return_tensors="pt", padding=True, truncation=True, max_length=128
        ).to(model.device)
        
        # Forward pass triggers hooks
        model(**enc)
        
        # Stack layers: [Batch, Layers, Heads, Dim]
        batch_acts = np.stack(layer_outputs, axis=1)
        activations.append(batch_acts)

    for h in hooks: h.remove()
    return np.concatenate(activations, axis=0)

# ==========================================
# 3. PARALLEL METRICS ENGINE (CPU)
# ==========================================
def calculate_metrics_for_single_head(head_data, party_labels):
    """
    Calculates all 5 metrics for a SINGLE head.
    This function will be mapped across 1024 heads in parallel.
    """
    # Filter Valid Data
    valid_mask = np.isin(party_labels, [100, 200])
    X = head_data[valid_mask]
    y = party_labels[valid_mask]
    
    # Centering for PCA/Covariance
    X_centered = X - np.mean(X, axis=0)
    
    results = {}
    
    # --- BLOCK A: PCA BASED METRICS (3 in 1) ---
    # We run PCA once to get eigenvalues, used for Dispersion, PC1, and Intrinsic Dim
    try:
        pca = PCA(n_components=10) # We only need top eigenvalues
        pca.fit(X_centered)
        evals = pca.explained_variance_
        
        # 3. Total Dispersion (Sum of variance/eigenvalues)
        results['Total_Dispersion'] = np.sum(evals)
        
        # 4. Explained Variance Ratio of PC1
        results['PC1_Ratio'] = pca.explained_variance_ratio_[0]
        
        # 5. Intrinsic Dimensionality (Participation Ratio)
        sum_evals = np.sum(evals)
        sum_sq_evals = np.sum(evals**2)
        if sum_sq_evals > 0:
            results['Intrinsic_Dim'] = (sum_evals**2) / sum_sq_evals
        else:
            results['Intrinsic_Dim'] = 0.0
            
    except Exception:
        results['Total_Dispersion'] = 0.0
        results['PC1_Ratio'] = 0.0
        results['Intrinsic_Dim'] = 0.0

    # --- BLOCK B: CLUSTER METRICS ---
    
    # 2. Davies-Bouldin Index
    # (Lower is better separation, so higher polarization means LOWER score usually)
    try:
        if len(np.unique(y)) > 1:
            results['Davies_Bouldin'] = davies_bouldin_score(X, y)
        else:
            results['Davies_Bouldin'] = 10.0 # Bad score
    except:
        results['Davies_Bouldin'] = 10.0

    # 1. Mahalanobis Distance
    try:
        dems = X[y == 100]
        reps = X[y == 200]
        
        if len(dems) > 5 and len(reps) > 5:
            # Pooled Covariance with regularization
            cov_pool = (np.cov(dems, rowvar=False) + np.cov(reps, rowvar=False)) / 2
            cov_pool += np.eye(cov_pool.shape[0]) * 1e-6 # Regularize
            
            inv_cov = inv(cov_pool)
            mu_d, mu_r = np.mean(dems, axis=0), np.mean(reps, axis=0)
            results['Mahalanobis'] = mahalanobis(mu_d, mu_r, inv_cov)
        else:
            results['Mahalanobis'] = 0.0
    except (LinAlgError, ValueError):
        results['Mahalanobis'] = 0.0
        
    return results

# ==========================================
# 4. MAIN EXECUTION LOOP
# ==========================================

# Setup
model, tokenizer = load_model(MODEL_PATH)

df_nom = pd.read_csv(NOMINATE_CSV)
df_nom = df_nom[df_nom['party_code'].isin([100, 200])].dropna(subset=['bioname'])
party_labels = df_nom['party_code'].values

full_results = []

print(f"\nStarting Optimized Pipeline on A100 (Batch Size {BATCH_SIZE})")
print(f"Parallel processing enabled for metrics calculation.")

for topic_name, topic_desc in TOPICS.items():
    t0 = time.time()
    print(f"\n--- Topic: {topic_name} ---")
    
    # 1. Generate Prompts
    prompts = []
    for name in df_nom['bioname']:
        if topic_desc:
            user_msg = (
                f"Generate a statement by {name} on {topic_desc}."
            )
        else:
            user_msg = (
                f"Generate a statement by {name}."
            )
        prompts.append(user_msg)
        
    # 2. Extract Heads (GPU Bound)
    # Shape: [N, 32, 32, 128]
    X_heads = extract_heads_batched(model, tokenizer, prompts, batch_size=BATCH_SIZE)
    
    # 3. Parallel Metrics (CPU Bound)
    # Flatten L and H to iterate easily: List of (1024) arrays of shape (N, 128)
    N, L, H, D = X_heads.shape
    flat_heads = [X_heads[:, l, h, :] for l in range(L) for h in range(H)]
    
    print("  > Computing metrics for 1024 heads (Parallel)...")
    
    # Uses all available CPU cores
    metrics_flat = Parallel(n_jobs=-1)(
        delayed(calculate_metrics_for_single_head)(head_data, party_labels) 
        for head_data in flat_heads
    )
    
    # 4. Aggregation & Storage
    # Reshape back to (32, 32) grids for visualization
    metric_grids = {k: np.zeros((L, H)) for k in metrics_flat[0].keys()}
    
    idx = 0
    for l in range(L):
        for h in range(H):
            m = metrics_flat[idx]
            for key in m:
                metric_grids[key][l, h] = m[key]
            idx += 1
            
    # Calculate Averages/Max for Summary
    summary = {"Topic": topic_name}
    for key, grid in metric_grids.items():
        summary[f"Avg_{key}"] = np.mean(grid)
        summary[f"Max_{key}"] = np.max(grid)
        # Store the full grid for heatmaps later
        summary[f"Grid_{key}"] = grid
        
    full_results.append(summary)
    print(f"  > Done in {time.time() - t0:.1f}s. Avg Mahalanobis: {summary['Avg_Mahalanobis']:.4f}")


/project/jevans/maxzhuyt/honest_llama_env/lib/python3.10/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/project/jevans/maxzhuyt/honest_llama_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model from: /project/jevans/maxzhuyt/models/Meta-Llama-3.1-8B-Instruct...


Loading checkpoint shards: 100%|██████████| 4/4 [00:05<00:00,  1.48s/it]



Starting Optimized Pipeline on A100 (Batch Size 128)
Parallel processing enabled for metrics calculation.

--- Topic: Baseline ---
  > Extracting with Batch Size 128...
  > Computing metrics for 1024 heads (Parallel)...
  > Done in 8.6s. Avg Mahalanobis: 1.7533

--- Topic: imm_unauth ---
  > Extracting with Batch Size 128...
  > Computing metrics for 1024 heads (Parallel)...
  > Done in 7.5s. Avg Mahalanobis: 1.8213

--- Topic: birthright ---
  > Extracting with Batch Size 128...
  > Computing metrics for 1024 heads (Parallel)...
  > Done in 7.5s. Avg Mahalanobis: 1.7965

--- Topic: illeg_child ---
  > Extracting with Batch Size 128...
  > Computing metrics for 1024 heads (Parallel)...
  > Done in 7.5s. Avg Mahalanobis: 1.8425

--- Topic: border_wall ---
  > Extracting with Batch Size 128...
  > Computing metrics for 1024 heads (Parallel)...
  > Done in 7.8s. Avg Mahalanobis: 1.7573

--- Topic: spend_border ---
  > Extracting with Batch Size 128...
  > Computing metrics for 1024 heads

In [2]:

# ==========================================
# 5. VISUALIZATION
# ==========================================
df_res = pd.DataFrame(full_results)

In [3]:
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
df_res.to_pickle(f"df_llm_anes_res_{timestamp}.pkl")

In [4]:
def calc_center_of_gravity(grid):
    """
    Calculate the center of gravity (weighted average layer index) for a 2D grid.
    Assumes axis 0 is layer, axis 1 is head.
    Returns a float representing the average layer number (0-31).
    """
    # Handle invalid grids
    if not isinstance(grid, np.ndarray):
        return np.nan
    
    # Replace any inf or nan values with 0
    grid = np.nan_to_num(grid, nan=0.0, posinf=0.0, neginf=0.0)
    
    L, H = grid.shape
    # Layer indices (0 to L-1, e.g., 0-31 for 32 layers)
    layer_idx = np.arange(L)
    # Sum polarization for each layer (across heads)
    layer_sum = grid.sum(axis=1)
    total = layer_sum.sum()
    if total == 0:
        return np.nan
    cog = np.sum(layer_idx * layer_sum) / total
    return cog

def safe_reciprocal(grid):
    """Compute reciprocal only where values are valid, avoiding divide-by-zero warnings."""
    if not isinstance(grid, np.ndarray):
        return np.nan
    result = np.full_like(grid, np.nan, dtype=float)
    mask = np.isfinite(grid) & (grid > 1e-10)
    result[mask] = 1.0 / grid[mask]
    return result

# Safely compute reciprocal, replacing inf/nan with nan
df_res['Grid_Davies_Bouldin_Rev'] = df_res['Grid_Davies_Bouldin'].apply(safe_reciprocal)
df_res["Polarization_CoG"] = df_res["Grid_Davies_Bouldin_Rev"].apply(calc_center_of_gravity)

In [5]:
df_res

,Topic,Avg_Total_Dispersion,Max_Total_Dispersion,Grid_Total_Dispersion,Avg_PC1_Ratio,Max_PC1_Ratio,Grid_PC1_Ratio,Avg_Intrinsic_Dim,Max_Intrinsic_Dim,Grid_Intrinsic_Dim,Avg_Davies_Bouldin,Max_Davies_Bouldin,Grid_Davies_Bouldin,Avg_Mahalanobis,Max_Mahalanobis,Grid_Mahalanobis,Grid_Davies_Bouldin_Rev,Polarization_CoG
0,Baseline,0.045554,1.448281,"[[0.0006188098923303187, 1.9228991732234135e-0...",0.427635,0.962262,"[[0.10159828513860703, 0.13500182330608368, 0....",3.596877,9.162354,"[[8.114928245544434, 6.813268184661865, 1.0635...",9.528041,28.719542,"[[10.783429817301958, 10.35005957492913, 8.014...",1.753346,2.739318,"[[1.2788956213312561, 0.6757404399234126, 0.29...","[[0.09273487349966382, 0.09661780135278568, 0....",15.590125
1,imm_unauth,0.027519,1.026740,"[[6.882594607304782e-05, 1.67380985658383e-05,...",0.383584,0.928955,"[[0.08229353278875351, 0.3883313834667206, 0.9...",3.910840,9.644713,"[[8.500543594360352, 2.608874559402466, 1.1215...",7.896980,23.347598,"[[11.055586626642517, 10.549779921743776, 8.34...",1.821281,2.709131,"[[0.958553771951725, 0.5467635974007394, 0.210...","[[0.09045200709568237, 0.0947887071974777, 0.1...",16.267675
2,birthright,0.029259,1.234827,"[[0.0001090360528905876, 1.4733432180946693e-0...",0.390944,0.926908,"[[0.07973366975784302, 0.2834092378616333, 0.8...",3.863772,9.575510,"[[8.538965225219727, 3.732590675354004, 1.2968...",8.756335,37.282318,"[[11.324156914608434, 10.980920159543873, 12.7...",1.796550,2.706335,"[[1.0605913931239994, 0.5686916985976951, 0.19...","[[0.08830679471687433, 0.09106704952506806, 0....",15.710361
3,illeg_child,0.034320,1.387254,"[[7.273071969393641e-05, 1.6665640941937454e-0...",0.404816,0.930948,"[[0.08218064904212952, 0.3767225742340088, 0.9...",3.750405,9.438087,"[[8.50680160522461, 2.7016663551330566, 1.1262...",7.118723,24.685416,"[[11.063780075533678, 10.626437881151228, 8.32...",1.842531,2.718196,"[[0.9693479596408837, 0.5499641820532404, 0.21...","[[0.09038502150014614, 0.09410491184197878, 0....",17.001089
4,border_wall,0.051774,1.831442,"[[2.631974348332733e-05, 2.040410618064925e-05...",0.433771,0.953440,"[[0.10917602479457855, 0.5284090638160706, 0.9...",3.521596,9.403046,"[[7.660077095031738, 1.853047251701355, 1.0807...",10.258293,37.082162,"[[12.744264751844689, 10.183267611893983, 37.0...",1.757255,2.689631,"[[0.7421900881263355, 0.468451204289215, 0.144...","[[0.07846666869151894, 0.0982003064352357, 0.0...",14.996041
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,biden_foreign,0.035813,1.346578,"[[4.419283868628554e-05, 1.816969597712159e-05...",0.387266,0.971476,"[[0.09414758533239365, 0.4620116651058197, 0.9...",3.932977,9.573914,"[[8.15821647644043, 2.1666510105133057, 1.0467...",8.580616,27.501570,"[[12.620560758757476, 9.961463210670438, 11.19...",1.818323,2.674582,"[[0.8200106296627833, 0.5219610377511005, 0.17...","[[0.07923578192087025, 0.10038685872260496, 0....",15.887371
60,israel_aid,0.032180,1.328017,"[[4.4061423977836967e-05, 1.7841579392552376e-...",0.399717,0.971078,"[[0.09411658346652985, 0.44776538014411926, 0....",3.828523,9.500568,"[[8.139799118041992, 2.2474186420440674, 1.047...",9.307558,26.529658,"[[12.555241662979606, 9.974290823994302, 11.34...",1.765294,2.653067,"[[0.8146404586550455, 0.5226618302532186, 0.17...","[[0.07964800892272753, 0.10025775442544599, 0....",15.529058
61,free_trade,0.029859,1.243908,"[[5.35774597665295e-05, 1.7485601347289048e-05...",0.391001,0.968897,"[[0.09123347699642181, 0.42694491147994995, 0....",3.875265,9.580939,"[[8.300579071044922, 2.369307279586792, 1.0508...",9.279105,26.480586,"[[11.496728658127076, 9.958945437016306, 9.277...",1.751102,2.699998,"[[0.8851983268589544, 0.5343680944309094, 0.22...","[[0.08698126482206717, 0.10041223805515691, 0....",15.572021
62,intl_trade_job,0.028545,1.111888,"[[1.2261183655937202e-05, 2.7019261324312538e-...",0.377772,1.000001,"[[0.15799424052238464, 0.5999829769134521, 0.9...",3.962967,9.658922,"[[6.36361408233

In [6]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
def adjust_text_positions(x, y, text_list, max_iterations=100, step_size=0.01):
    """
    Iteratively moves text away from:
      1. Its own data point (repulsion)
      2. Other text labels (collision avoidance)
    Returns optimized x_text, y_text arrays.
    """
    n = len(x)
    # Start text positions exactly at data points
    tx = np.array(x, dtype=float)
    ty = np.array(y, dtype=float)
    
    # Normalize coordinates to 0-1 scale for consistent 'force' calculations
    # We will map them back later.
    x_min, x_max = np.min(x), np.max(x)
    y_min, y_max = np.min(y), np.max(y)
    x_range = x_max - x_min
    y_range = y_max - y_min
    
    # Avoid division by zero
    if x_range == 0: x_range = 1
    if y_range == 0: y_range = 1
    
    tx_norm = (tx - x_min) / x_range
    ty_norm = (ty - y_min) / y_range
    x_norm = (np.array(x) - x_min) / x_range
    y_norm = (np.array(y) - y_min) / y_range

    for _ in range(max_iterations):
        # Calculate forces
        grad_x = np.zeros(n)
        grad_y = np.zeros(n)
        
        for i in range(n):
            # 1. Force pushing text away from its own data point
            # We want it slightly offset, not ON TOP
            dist_self_x = tx_norm[i] - x_norm[i]
            dist_self_y = ty_norm[i] - y_norm[i]
            dist_sq = dist_self_x**2 + dist_self_y**2
            
            # If too close to dot, push away (standard radius)
            target_radius = 0.04 # 4% of plot width
            if dist_sq < target_radius**2:
                 # Push randomly if exactly on top, otherwise radially
                if dist_sq == 0:
                    grad_x[i] += (np.random.random() - 0.5) * 0.1
                    grad_y[i] += (np.random.random() - 0.5) * 0.1
                else:
                    force = (target_radius - np.sqrt(dist_sq)) 
                    grad_x[i] += force * (dist_self_x / np.sqrt(dist_sq))
                    grad_y[i] += force * (dist_self_y / np.sqrt(dist_sq))

            # 2. Force pushing text away from OTHER labels
            for j in range(n):
                if i == j: continue
                
                diff_x = tx_norm[i] - tx_norm[j]
                diff_y = ty_norm[i] - ty_norm[j]
                dist_sq = diff_x**2 + diff_y**2
                
                # Collision radius (approximate text box size)
                min_dist = 0.05 # 5% of plot width
                
                if dist_sq < min_dist**2:
                     if dist_sq == 0:
                        grad_x[i] += (np.random.random() - 0.5) * 0.1
                        grad_y[i] += (np.random.random() - 0.5) * 0.1
                     else:
                        force = (min_dist - np.sqrt(dist_sq)) * 2 # Stronger force for text collision
                        grad_x[i] += force * (diff_x / np.sqrt(dist_sq))
                        grad_y[i] += force * (diff_y / np.sqrt(dist_sq))
                        
        # Apply movements
        tx_norm += grad_x * step_size
        ty_norm += grad_y * step_size
        
        # Clamp to 0-1 to keep inside plot (optional)
        tx_norm = np.clip(tx_norm, 0, 1)
        ty_norm = np.clip(ty_norm, 0, 1)

    # Convert back to original scale
    final_tx = tx_norm * x_range + x_min
    final_ty = ty_norm * y_range + y_min
    
    return final_tx, final_ty

def plot(df_plot):
    # Run the physics engine to get new coordinates for the TEXT ONLY
    # (Assuming adjust_text_positions is defined in your environment)
    new_x, new_y = adjust_text_positions(
        df_plot["Avg_Mahalanobis"].values, 
        df_plot["Avg_Total_Dispersion"].values, 
        df_plot["Topic"].values,
        max_iterations=200, 
        step_size=0.5
    )

    df_plot['text_x'] = new_x
    df_plot['text_y'] = new_y

    # --- 3. PLOTTING (Using Graph Objects for Layering) ---

    fig = go.Figure()

    # Layer 1: The Dots (Markers)
    unique_topics = df_plot['Topic'].unique()
    colors = px.colors.qualitative.Plotly 

    for i, topic in enumerate(unique_topics):
        df_sub = df_plot[df_plot['Topic'] == topic]
        color = colors[i % len(colors)]
        
        fig.add_trace(go.Scatter(
            x=df_sub['Avg_Mahalanobis'],
            y=df_sub['Avg_Total_Dispersion'],
            mode='markers',
            name=topic,
            # Pass Intrinsic Dim as custom data for the hover
            customdata=df_sub['Avg_Intrinsic_Dim'], 
            marker=dict(
                size=14,  # FIXED SIZE for visualization
                color=color,
                opacity=0.85,
                line=dict(width=1, color='DarkSlateGrey')
            ),
            hovertemplate=(
                "<b>%{text}</b><br>" +
                "Polarization: %{x:.4f}<br>" +
                "Intensity: %{y:.2f}<br>" +
                "Intrinsic Dim: %{customdata:.2f}<br>" + # Added here
                "<extra></extra>" 
            ),
            text=df_sub['Topic']
        ))

    # Layer 2: The Lines (Connecting dots to labels)
    for i, row in df_plot.iterrows():
        fig.add_trace(go.Scatter(
            x=[row['Avg_Mahalanobis'], row['text_x']],
            y=[row['Avg_Total_Dispersion'], row['text_y']],
            mode='lines',
            line=dict(color='grey', width=0.3),
            showlegend=False,
            hoverinfo='skip'
        ))

    # Layer 3: The Text (at new optimized positions)
    fig.add_trace(go.Scatter(
        x=df_plot['text_x'],
        y=df_plot['text_y'],
        mode='text',
        text=df_plot['Topic'],
        textfont=dict(size=11, color='black'),
        showlegend=False,
        hoverinfo='skip'
    ))

    # --- 4. LAYOUT & ANNOTATIONS ---

    fig.update_layout(
        title="<b>What Politicans Say about Everyday Topics</b><br>X: Polarization | Y: Total Disagreement",
        template="plotly_white",
        height=800,
        width=1000,
        xaxis_title="Polarization (Mahalanobis Distance)",
        yaxis_title="Total Discourse Disagreement (Dispersion)",
        showlegend=True
    )

    # Add Quadrants
    mid_x = df_plot['Avg_Mahalanobis'].median()
    mid_y = df_plot['Avg_Total_Dispersion'].median()

    fig.add_hline(y=mid_y, line_dash="dot", line_color="grey", opacity=0.5)
    fig.add_vline(x=mid_x, line_dash="dot", line_color="grey", opacity=0.5)

    # Add Quadrant Labels (Watermarks)
    fig.add_annotation(x=df_plot['Avg_Mahalanobis'].max(), y=df_plot['Avg_Total_Dispersion'].max(),
                    text="<b>High Polarization<br>High Variance</b>", showarrow=False, align="right", opacity=0.3,
                    xref="x", yref="y", xanchor="right", yanchor="top")
    fig.add_annotation(x=df_plot['Avg_Mahalanobis'].min(), y=df_plot['Avg_Total_Dispersion'].max(),
                    text="<b>Low Polarization<br>High Variance</b>", showarrow=False, align="left", opacity=0.3,
                    xref="x", yref="y", xanchor="left", yanchor="top")
    fig.add_annotation(x=df_plot['Avg_Mahalanobis'].max(), y=df_plot['Avg_Total_Dispersion'].min(),
                    text="<b>High Polarization<br>Low Variance</b>", showarrow=False, align="right", opacity=0.3,
                    xref="x", yref="y", xanchor="right", yanchor="bottom")
    fig.add_annotation(x=df_plot['Avg_Mahalanobis'].min(), y=df_plot['Avg_Total_Dispersion'].min(),
                    text="<b>Low Polarization<br>Low Variance</b>", showarrow=False, align="left", opacity=0.3,
                    xref="x", yref="y", xanchor="left", yanchor="bottom")

    fig.show()
    fig.write_html("us_cultural_geometry_all.html", include_plotlyjs="cdn")

In [7]:
plot(df_res)

In [8]:
# Drop topics that are strongly related to the 2020 election cycle. 
DROP_TOPICS = {
    "rus_interf",
    "urban_unrest",
    "econ_now",
    "trump_corr",
    "border_wall"
}

df_res = df_res[~df_res["Topic"].isin(DROP_TOPICS)]
# delete all topics with "ft" in the name



In [9]:
anes_path = "policy_polarization.csv"

df_anes = pd.read_csv(anes_path)
df_anes = df_anes.rename(columns={"issue": "Topic", "mahalanobis_distance": "ANES_Mahalanobis", "variance": "ANES_Variance"})

## Comparing Party Polarization between LLM and ANES

In [10]:

# Keep only Topic + Avg_Mahalanobis from LLM results
# rename columns

df_llm = df_res[["Topic", "Avg_Mahalanobis", "Avg_Total_Dispersion", "Polarization_CoG"]].copy()

# Inner join ensures exact key matching
df_merged = df_llm.merge(
    df_anes,
    left_on="Topic",
    right_on="Topic",
    how="inner"
)

pearson_corr = df_merged["Avg_Mahalanobis"].corr(
    df_merged["ANES_Mahalanobis"],
    method="pearson"
)

print(f"Pearson correlation: {pearson_corr:.3f}")
spearman_corr = df_merged["Avg_Mahalanobis"].corr(
    df_merged["ANES_Mahalanobis"],
    method="spearman"
)

print(f"Spearman correlation: {spearman_corr:.3f}")
df_merged["LLM_rank"] = df_merged["Avg_Mahalanobis"].rank(ascending=False)
df_merged["ANES_rank"] = df_merged["ANES_Mahalanobis"].rank(ascending=False)

df_merged.sort_values("Avg_Mahalanobis", ascending=False)


Pearson correlation: 0.325
Spearman correlation: 0.344


,Topic,Avg_Mahalanobis,Avg_Total_Dispersion,Polarization_CoG,ANES_Mahalanobis,ANES_Variance,area,explanation,LLM_rank,ANES_rank
43,ft_trans,1.897050,0.042290,18.264347,1.069882,0.076328,Transgender,Feeling thermometer rating toward transgender ...,1.0,26.0
41,trans_bath,1.884847,0.033458,18.921809,1.276135,0.161220,Transgender,Views on transgender bathroom access policies.,2.0,15.0
5,paid_leave,1.851979,0.025422,16.712534,0.713889,0.072350,Labor,Support for paid parental/family leave policie...,3.0,43.0
2,illeg_child,1.842531,0.034320,17.001089,0.948134,0.075581,Immigration,Views on how the U.S. should handle children o...,4.0,34.0
44,lg_job,1.841690,0.036985,17.579029,0.602085,0.088672,Lesbian/Gay,Views on legal protections against job discrim...,5.0,46.5
48,clim_imp,1.833248,0.037742,16.992993,1.565396,0.114112,Climate,Importance assigned to climate change as an is...,6.0,6.0
21,biden_crime,1.832470,0.032374,16.263938,1.923465,0.160533,CrimeAndPolicing,Approval of Joe Biden's handling of crime.,7.0,2.0
45,lg_marry,1.827036,0.034325,16.754018,0.602085,0.088672,Lesbian/Gay,Support for same-sex marriage / marriage right...,8.0,46.5
16,ar_ban,1.824237,0.024441,16.684285,0.940938,0.157338,Guns,Support for banning assault rifles/assault wea...,9.0,35.0
8,ft_union,1.822874,0.034088,16.232798,0.963094,0.057614,Labor,Feeling thermometer rating toward labor unions.,10.0,33.0


In [11]:
unique_areas = df_merged['area'].unique()
unique_areas

array(['Immigration', 'Labor', 'Institutions', 'Elections', 'Guns',
       'CrimeAndPolicing', 'Abortion', 'Health', 'Education',
       'Redistribution', 'Diversity', 'Transgender', 'Lesbian/Gay',
       'Gender', 'Climate', 'ForeignPolicy', 'Trade'], dtype=object)

### Can we look at what subset correlate well and try to reverse engineer what might be the issue ...?

In [ ]:
# Sample subsets and find best correlations
import random
random.seed(42)
np.random.seed(42)

n_iterations = 1000000
results = []
import numpy as np
import pandas as pd
from itertools import product

# Calculate total possible combinations
counts = df_merged.groupby('area')['Topic'].nunique().values
total_possible = np.prod(counts)

print(f"Total possible unique combinations: {total_possible:,}")
print(f"Sampling {n_iterations:,} iterations represents {(n_iterations/total_possible)*100:.4f}% of the search space.")
import random

# 1. Pre-process data for speed
# Create a list of (Avg_Mahalanobis, ANES_Mahalanobis) arrays, one per area
area_data = []
area_topic_names = []
for area in df_merged['area'].unique():
    subset = df_merged[df_merged['area'] == area]
    # We store the values as a numpy array for fast indexing
    area_data.append(subset[["Avg_Mahalanobis", "ANES_Mahalanobis"]].values)
    area_topic_names.append(subset["Topic"].values)

n_areas = len(area_data)

# 2. Pre-generate all random selections (Vectorized sampling)
# This creates a matrix of shape (n_iterations, n_areas)
random_indices = [np.random.randint(0, len(data), n_iterations) for data in area_data]

results = []

# 3. Optimized Loop
for i in range(n_iterations):
    # Extract the pre-selected values for this iteration
    current_sample = np.array([area_data[j][random_indices[j][i]] for j in range(n_areas)])
    
    # current_sample is now (n_areas, 2)
    x = current_sample[:, 0]
    y = current_sample[:, 1]
    
    # Fast correlation (Series objects have overhead, numpy is faster)
    pearson = np.corrcoef(x, y)[0, 1]
    
    # For Spearman, we still use pandas or scipy as numpy doesn't have a direct equivalent
    # But we only do it if we need it to save time
    results.append({
        'iteration': i,
        'pearson': pearson,
        'indices': [random_indices[j][i] for j in range(n_areas)] 
    })

# 4. Post-process top results to get names
df_results = pd.DataFrame(results).sort_values('pearson', ascending=False)

In [20]:
top_10 = df_results.head(30).copy()

# Recover topic names only for the top 10 to save computation
final_output = []
for _, row in top_10.iterrows():
    topics = [area_topic_names[j][row['indices'][j]] for j in range(n_areas)]
    final_output.append({
        'iteration': row['iteration'],
        'pearson': row['pearson'],
        'topics': topics
    })

# Print results
for res in final_output:
    print(f"Iteration {res['iteration']}: Pearson={res['pearson']:.3f}")
    print(f"Topics: {', '.join(res['topics'])}\n")

Iteration 608647: Pearson=0.897
Topics: birthright, min_wage, checks_power, vote_denied, gun_bkg_chk, crime_spend, scotus_abort, vax_school, affirm_action, spend_welfare, assist_black, trans_military, lg_refuse_service, ft_fem, clim_imp, biden_foreign, intl_trade_job

Iteration 508401: Pearson=0.892
Topics: birthright, min_wage, checks_power, vote_denied, gun_bkg_chk, biden_crime, scotus_abort, vax_school, affirm_action, spend_welfare, discuss_race, trans_military, lg_refuse_service, ft_fem, ghg_emiss, biden_foreign, intl_trade_job

Iteration 325892: Pearson=0.884
Topics: birthright, min_wage, checks_power, vote_denied, gun_bkg_chk, biden_crime, abortion, vax_school, spend_school, spend_poor, assist_black, trans_military, lg_refuse_service, ft_fem, ghg_emiss, biden_foreign, free_trade

Iteration 138975: Pearson=0.872
Topics: birthright, min_wage, ft_sci, vote_denied, gun_bkg_chk, biden_crime, scotus_abort, vax_school, spend_school, spend_welfare, black_favor, trans_military, lg_refuse_

In [13]:
df_merged

,Topic,Avg_Mahalanobis,Avg_Total_Dispersion,Polarization_CoG,ANES_Mahalanobis,ANES_Variance,area,explanation,LLM_rank,ANES_rank
0,imm_unauth,1.821281,0.027519,16.267675,1.007885,0.087450,Immigration,Policy preferences regarding unauthorized/undo...,11.0,30.0
1,birthright,1.796550,0.029259,15.710361,1.036311,0.129074,Immigration,Support for birthright citizenship for childre...,24.0,28.0
2,illeg_child,1.842531,0.034320,17.001089,0.948134,0.075581,Immigration,Views on how the U.S. should handle children o...,4.0,34.0
3,spend_border,1.763452,0.024095,15.967041,1.550767,0.113009,Immigration,Preferences for federal spending levels on bor...,33.0,8.0
4,immig_levels,1.784536,0.033639,15.687273,0.699658,0.088596,Immigration,Opinion on whether the number of immigrants pe...,27.0,44.0
5,paid_leave,1.851979,0.025422,16.712534,0.713889,0.072350,Labor,Support for paid parental/family leave policie...,3.0,43.0
6,job_gov_guar,1.727385,0.023750,16.172051,1.478903,0.105467,Labor,Support for a government-guaranteed job progra...,44.0,10.0
7,min_wage,1.794113,0.028318,15.996795,1.008874,0.071326,Labor,Support for raising the minimum wage / minimum...,25.0,29.0
8,ft_union,1.822874,0.034088,16.232798,0.963094,0.057614,Labor,Feeling thermometer rating toward labor unions.,10.0,33.0
9,journ_access,1.690958,0.022178,16.240637,0.834154,0.104073,Institutions,Support for press/journalist access to governm...,57.0,41.0


In [14]:
def compare_standardized_columns(df, col1, col2, group_by='area'):
    """
    Demean and standardize two columns separately, calculate their difference,
    and group by specified column.
    
    Parameters:
    - df: DataFrame containing the data
    - col1: First column name to standardize
    - col2: Second column name to standardize
    - group_by: Column name to group results by (default 'area')
    
    Returns:
    - Modified DataFrame with normalized columns and difference
    """
    # Create a copy to avoid modifying original
    df_work = df.copy()
    
    # Demean and standardize each column separately
    mean1 = df_work[col1].mean()
    std1 = df_work[col1].std()
    mean2 = df_work[col2].mean()
    std2 = df_work[col2].std()
    
    col1_norm = col1 + '_norm'
    col2_norm = col2 + '_norm'
    
    df_work[col1_norm] = (df_work[col1] - mean1) / std1
    df_work[col2_norm] = (df_work[col2] - mean2) / std2
    
    # Calculate difference between standardized columns
    df_work['diff'] = df_work[col1_norm] - df_work[col2_norm]
    
    # Calculate mean differences by group
    group_diff = df_work.groupby(group_by).agg({
        'diff': ['mean', 'count']
    }).round(4)
    
    group_diff.columns = ['mean_diff', 'n_topics']
    group_diff = group_diff.sort_values('mean_diff', ascending=False)
    
    print(f"Standardized Differences by {group_by}:")
    print(group_diff)
    
    print("\nDetailed breakdown:")
    print(df_work[['Topic', group_by, col1_norm, col2_norm, 'diff', "Polarization_CoG"]].sort_values('diff', ascending=False).to_markdown())
    
    return df_work

# Compare Avg_Total_Dispersion (LLM) with ANES_Variance
df_diff = compare_standardized_columns(df_merged, 'Avg_Mahalanobis', 'ANES_Mahalanobis', group_by='area')

Standardized Differences by area:
                  mean_diff  n_topics
area                                 
Lesbian/Gay          1.3792         3
Transgender          1.3398         3
Guns                 0.9566         3
Immigration          0.4688         5
Labor                0.4332         4
Trade                0.2412         3
Abortion             0.0652         2
Gender              -0.0109         1
ForeignPolicy       -0.0935         4
Diversity           -0.1059         5
Education           -0.1789         3
Climate             -0.3879         3
Institutions        -0.4209         3
Redistribution      -0.5810         4
Elections           -0.6607         3
CrimeAndPolicing    -0.7436         5
Health              -1.0140         4

Detailed breakdown:
|    | Topic             | area             |   Avg_Mahalanobis_norm |   ANES_Mahalanobis_norm |        diff |   Polarization_CoG |
|---:|:------------------|:-----------------|-----------------------:|---------------------

In [15]:
corr = df_diff['diff'].corr(df_diff['Polarization_CoG'], method='pearson')
print(f"Pearson correlation between standardized difference and Polarization_CoG: {corr:.3f}")


Pearson correlation between standardized difference and Polarization_CoG: 0.356


In [16]:
# restrict area to some areas
df_subset = df_merged[df_merged['area'].isin(['Policing', 'Labor', "Immigration", "Education", "Redistribution"])]
pearson_corr_subset = df_subset["Avg_Mahalanobis"].corr(
    df_subset["ANES_Mahalanobis"],
    method="pearson"
)
print(f"\nSubset Pearson correlation: {pearson_corr_subset:.3f}")
spearman_corr_subset = df_subset["Avg_Mahalanobis"].corr(
    df_subset["ANES_Mahalanobis"],
    method="spearman"
)
print(f"Subset Spearman correlation: {spearman_corr_subset:.3f}")



Subset Pearson correlation: -0.674
Subset Spearman correlation: -0.594


## Comparing Total Variance between ANES and LLM

In [17]:

pearson_corr = df_merged["Avg_Total_Dispersion"].corr(
    df_merged["ANES_Variance"],
    method="pearson"
)

print(f"Pearson correlation: {pearson_corr:.3f}")
spearman_corr = df_merged["Avg_Total_Dispersion"].corr(
    df_merged["ANES_Variance"],
    method="spearman"
)

print(f"Spearman correlation: {spearman_corr:.3f}")
df_merged["LLM_rank"] = df_merged["Avg_Total_Dispersion"].rank(ascending=False)
df_merged["ANES_rank"] = df_merged["ANES_Variance"].rank(ascending=False)

df_merged.sort_values("Avg_Total_Dispersion", ascending=False)

Pearson correlation: -0.147
Spearman correlation: -0.056


,Topic,Avg_Mahalanobis,Avg_Total_Dispersion,Polarization_CoG,ANES_Mahalanobis,ANES_Variance,area,explanation,LLM_rank,ANES_rank
47,ft_fem,1.811181,0.046814,16.600159,1.355457,0.071815,Gender,Feeling thermometer rating toward feminists.,1.0,45.0
39,ft_asian,1.798940,0.045398,17.394634,0.222862,0.043342,Diversity,Feeling thermometer rating toward Asian-Americ...,2.0,57.0
43,ft_trans,1.897050,0.042290,18.264347,1.069882,0.076328,Transgender,Feeling thermometer rating toward transgender ...,3.0,41.0
20,ft_police,1.757427,0.040563,15.527693,1.106227,0.063131,CrimeAndPolicing,Feeling thermometer rating toward police.,4.0,49.0
11,ft_sci,1.772369,0.040353,15.896984,0.922126,0.040673,Institutions,Feeling thermometer rating toward scientists.,5.0,58.0
48,clim_imp,1.833248,0.037742,16.992993,1.565396,0.114112,Climate,Importance assigned to climate change as an is...,6.0,17.0
44,lg_job,1.841690,0.036985,17.579029,0.602085,0.088672,Lesbian/Gay,Views on legal protections against job discrim...,7.0,31.5
53,biden_foreign,1.818323,0.035813,15.887371,1.700641,0.173603,ForeignPolicy,Approval of Joe Biden's handling of foreign re...,8.0,2.0
45,lg_marry,1.827036,0.034325,16.754018,0.602085,0.088672,Lesbian/Gay,Support for same-sex marriage / marriage right...,9.0,31.5
2,illeg_child,1.842531,0.034320,17.001089,0.948134,0.075581,Immigration,Views on how the U.S. should handle children o...,10.0,42.0


In [18]:
df_merged = compare_standardized_columns(df_merged, 'Avg_Total_Dispersion', 'ANES_Variance')

Standardized Differences by area:
                  mean_diff  n_topics
area                                 
Gender               3.5735         1
Institutions         0.8501         3
Transgender          0.7636         3
Diversity            0.7272         5
Labor                0.4213         4
Lesbian/Gay          0.2696         3
ForeignPolicy        0.2048         4
Climate              0.1978         3
Immigration          0.1265         5
Trade               -0.0300         3
Abortion            -0.3677         2
Redistribution      -0.5813         4
Education           -0.6923         3
Elections           -0.6938         3
CrimeAndPolicing    -0.7274         5
Guns                -0.7390         3
Health              -0.8567         4

Detailed breakdown:
|    | Topic             | area             |   Avg_Total_Dispersion_norm |   ANES_Variance_norm |       diff |   Polarization_CoG |
|---:|:------------------|:-----------------|----------------------------:|---------------